adhoc = 'Yes'

In [31]:
adhoc: str 
n: int # utilized for looping through months

StatementMeta(, 9ad55c41-31ed-41ee-b2e3-cd10747e3bab, 33, Finished, Available, Finished, False)

In [32]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
import json
from pyspark.sql.functions import current_timestamp, trunc, add_months, date_format, col
import pytz
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import col, explode
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType
from pyspark.sql.functions import col, lit, struct
from pyspark.sql.types import StructType, StructField, StringType, IntegerType , DateType , BooleanType , DoubleType ,TimestampType,ArrayType,ArrayType,LongType
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DoubleType
from pyspark.sql.utils import *
#SET spark.sql.legacy.ctePrecedencePolicy = CORRECTED;
spark = SparkSession.builder.appName("json-to-parquet").config("spark.driver.maxResultSize", "-1").getOrCreate()
from pyspark.sql.utils import AnalysisException
from delta.tables import DeltaTable
import pandas as pd

StatementMeta(, 9ad55c41-31ed-41ee-b2e3-cd10747e3bab, 34, Finished, Available, Finished, False)

In [33]:
# Get current time in Eastern Time
eastern = pytz.timezone("US/Eastern")
now_et = datetime.now(eastern)

# # Get the first day of the previous month
# first_day_last_month = now_et.replace(day=1) - timedelta(days=1)
# start_date_prev_month = first_day_last_month.replace(day=1)

# # Get the last day of the previous month
# end_date_prev_month = first_day_last_month

StatementMeta(, 9ad55c41-31ed-41ee-b2e3-cd10747e3bab, 35, Finished, Available, Finished, False)

In [34]:
#
# if adhoc == 'No':
#     start_date = start_date_prev_month.strftime('%Y-%m-%d')
#     end_date = end_date_prev_month.strftime('%Y-%m-%d')
# else:
#     start_date = now_et.replace(day=1).strftime('%Y-%m-%d')
#     end_date=str(rundate)
# print(start_date)
# print(end_date)

StatementMeta(, 9ad55c41-31ed-41ee-b2e3-cd10747e3bab, 36, Finished, Available, Finished, False)

In [35]:
eastern = pytz.timezone("US/Eastern")
current_datetime=datetime.now(eastern)
# Split into date and time
rundate = current_datetime.date()
runtime = current_datetime.time()

print("Run date:", rundate)
print("Run time:", runtime)

StatementMeta(, 9ad55c41-31ed-41ee-b2e3-cd10747e3bab, 37, Finished, Available, Finished, False)

Run date: 2026-04-06
Run time: 15:40:38.858286


In [36]:
financial = spark.sql('select * from PROD_DEC_IH.financial')
financial_type = spark.sql('select * from PROD_DEC_IH.financial_type')
general_codes = spark.sql('select * from PROD_DEC_IH.general_codes')
Coverage_codes = spark.sql('select * from PROD_DEC_IH.Coverage_codes')

StatementMeta(, 9ad55c41-31ed-41ee-b2e3-cd10747e3bab, 38, Finished, Available, Finished, False)

In [37]:
Commissions = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/AgentCommission_versions'
PolicyCovPrem = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/Risks_Vehicles_PremiumFactors_versions'
PolicyTransactions = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/Transaction_versions'
Policy_versions = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/Policy_versions'
policyfees = 'abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/FeesAndTaxes_versions'


# Read data into DataFrames
Commissions = spark.read.format("delta").load(Commissions)
PolicyCovPrem = spark.read.format("delta").load(PolicyCovPrem)
PolicyTransactions=spark.read.format("delta").load(PolicyTransactions)
Policy_versions=spark.read.format("delta").load(Policy_versions)
policyfees=spark.read.format("delta").load(policyfees)




# Drop views
spark.catalog.dropTempView("Commissions")
spark.catalog.dropTempView("financial")
spark.catalog.dropTempView("financial_type")
spark.catalog.dropTempView("general_codes")
spark.catalog.dropTempView("Coverage_codes")
spark.catalog.dropTempView("PolicyCovPrem")
spark.catalog.dropTempView("PolicyTransactions")
spark.catalog.dropTempView("Policy_versions")
spark.catalog.dropTempView("policyfees")

# Recreate views
Commissions.createOrReplaceTempView("Commissions")
financial.createOrReplaceTempView("financial")
financial_type.createOrReplaceTempView("financial_type")
general_codes.createOrReplaceTempView("general_codes")
Coverage_codes.createOrReplaceTempView("Coverage_codes")
PolicyCovPrem.createOrReplaceTempView("PolicyCovPrem")
PolicyTransactions.createOrReplaceTempView("PolicyTransactions")
Policy_versions.createOrReplaceTempView("Policy_versions")
policyfees.createOrReplaceTempView("policyfees")


StatementMeta(, 9ad55c41-31ed-41ee-b2e3-cd10747e3bab, 39, Finished, Available, Finished, False)

In [38]:
spark.catalog.clearCache()

StatementMeta(, 9ad55c41-31ed-41ee-b2e3-cd10747e3bab, 40, Finished, Available, Finished, False)

In [39]:
def run_monthly_process(start_date, end_date): #Function to get monthly data for given date range and data source
	query = f'''
	with 
	agent_comm as 
	(select distinct policy_ref,value,commissiontype
	from Commissions where value>0 and transactiontype='NEW'
	),

	policy_data as(
	select policynumber,cov.name as coverage,
	'40' as company,'PA' as policy_type,SUBSTRING(policynumber,1,2) as policy_state,
	sum(cov.EffectivePremium) as netwritten,
	SUM(CASE WHEN t.type <> 'Cancellation' 
		THEN cov.EffectivePremium 
		ELSE 0 
		END )as grosswritten_by_policy_cov,
	SUM(IFNULL(cov.EffectivePremium*0.01*ag.Value,0)) as commission

	from Policy_versions p

	inner join PolicyTransactions t
	on t.policy_ref=p.policy_ref

	inner join PolicyCovPrem cov
	on cov.policy_ref=t.policy_ref
	and cov.Name in ('BI','PD','UMBI','UMPD','MP','FFH')

	left join agent_comm ag
	on ag.policy_ref=cov.policy_ref
	and ag.CommissionType in ('LBO')

	where t.status='Committed'  --## Considering only Committed Transactions ##
	AND (
    -- ## Block 1: Date is after EffectiveDate and within range ##
    (t.Date > t.EffectiveDate AND t.Date >= '{start_date}' AND t.Date < '{end_date}')
    OR 
    -- ## Block 2: EffectiveDate is in range and Date hasn't passed the end ##
    (t.EffectiveDate >= '{start_date}' AND t.EffectiveDate < '{end_date}' AND t.Date < '{end_date}')
)	-- ## Additional Date comparision block in line with business expectation ##
	--and p.policynumber in ('GAPA007709305','GAPA007709306')

	group by policynumber,cov.name

	union

	select policynumber,cov.name as coverage,
	'40' as company,'PA' as policy_type,SUBSTRING(policynumber,1,2) as policy_state,
	sum(cov.EffectivePremium) as netwritten,
	SUM(CASE WHEN t.type <> 'Cancellation' 
		THEN cov.EffectivePremium 
		ELSE 0 
		END )as grosswritten_by_policy_cov,
	SUM(IFNULL(cov.EffectivePremium*0.01*ag.Value,0)) as commission

	from Policy_versions p

	inner join PolicyTransactions t
	on t.policy_ref=p.policy_ref

	inner join PolicyCovPrem cov
	on cov.policy_ref=t.policy_ref
	and cov.Name in ('CMP','COL','TOW','RNT')

	left join agent_comm ag
	on ag.policy_ref=cov.policy_ref
	and ag.CommissionType in ('LBP')

	WHERE t.status = 'Committed' -- ## Considering only Committed Transactions ##
	AND (
    -- ## Block 1: Date is after EffectiveDate and within range ##
    (t.Date > t.EffectiveDate AND t.Date >= '{start_date}' AND t.Date < '{end_date}')
    OR 
    -- ## Block 2: EffectiveDate is in range and Date hasn't passed the end ##
    (t.EffectiveDate >= '{start_date}' AND t.EffectiveDate < '{end_date}' AND t.Date < '{end_date}')
	) -- ## Additional Date comparision block in line with business expectation ##
	--and p.policynumber in ('GAPA007709305','GAPA007709306')

	group by policynumber,cov.name

	union

	select policynumber,cov.name as coverage,
	'40' as company,'PA' as policy_type,SUBSTRING(policynumber,1,2) as policy_state,
	0.00 as netwritten,
	0.00 as grosswritten_by_policy_cov,
	sum(cov.EffectivePremium*0.01*ag.Value) as commission

	from Policy_versions p

	inner join PolicyTransactions t
	on t.policy_ref=p.policy_ref

	inner join PolicyCovPrem cov
	on cov.policy_ref=t.policy_ref

	inner join agent_comm ag
	on ag.policy_ref=cov.policy_ref
	and ag.CommissionType=cov.name
	and ag.CommissionType not in ('LBO','LBP')

	WHERE t.status = 'Committed' -- ## Considering only Committed Transactions ##
	AND (
    -- ## Block 1: Date is after EffectiveDate and within range ##
    (t.Date > t.EffectiveDate AND t.Date >= '{start_date}' AND t.Date < '{end_date}')
    OR 
    -- ## Block 2: EffectiveDate is in range and Date hasn't passed the end ##
    (t.EffectiveDate >= '{start_date}' AND t.EffectiveDate < '{end_date}' AND t.Date < '{end_date}')
	) --## Additional Date comparision block added in line with business expectation ##
	--and p.policynumber in ('GAPA007709305','GAPA007709306')

	group by policynumber,cov.name
	),

	claims_data as(
	select  cov.cov_stat_code,f.finc_clmc_cov_code, finc_clm_policy_pgm, finc_clm_lob, finc_clm_policy_st,
		IFNULL(SUM((
				CASE 
						WHEN fint_categ = 'P' THEN finc_amnt
						WHEN fint_categ = '1' THEN -finc_amnt
						ELSE 0
				END
			)), 0) -
			--subtracting recoveries from paid loss
				IFNULL(SUM((CASE 
						WHEN fint_categ = '3' THEN -finc_amnt
						WHEN fint_categ = 'S'  THEN finc_amnt
								ELSE 0
							END)), 0) -
				IFNULL(SUM((CASE 
						WHEN fint_categ = '4' THEN -finc_amnt
						WHEN fint_categ = 'X' THEN finc_amnt
								ELSE 0
							END)), 0) - 
				IFNULL(SUM((CASE 
						WHEN fint_categ = '5' THEN -finc_amnt
						WHEN fint_categ = 'Y' THEN finc_amnt
								ELSE 0
							END)), 0)
				AS paid_loss
			,

				IFNULL(SUM(CASE 
								WHEN fint_categ = 'E' THEN finc_amnt
									WHEN fint_categ = '2' THEN -finc_amnt
									ELSE 0
								END), 0)
				AS paid_expense

	FROM 
			financial f      
			inner join (select distinct policynumber from policy_data)p
			on p.policynumber=f.finc_policy
			INNER JOIN financial_type ft ON f.finc_opr_type = ft.fint_type
			INNER JOIN general_codes g ON g.gcp_code = f.finc_clm_policy_pgm
			left join Coverage_codes cov on f.finc_clmc_cov_code=cov.cov_code
		WHERE

			finc_acc_date >= '{start_date}' and finc_acc_date < '{end_date}'  --## Between being replaced with current logic to avoid missing on edge case transactions ##         
			AND gcp_internal_type = 'COMPANY' 
			AND gcp_valid_code = '1'
			AND gcp_valid_type = '1'
			AND f.finc_sup_valid = '1'
			AND f.finc_rev_trans_flg = '0'
			AND (f.finc_opr_approval_status = 'A' OR f.finc_opr_approval_status IS NULL)
		GROUP BY cov.cov_stat_code,
			f.finc_clmc_cov_code, 
			f.finc_clm_policy_pgm, 
			f.finc_clm_lob,
			f.finc_clm_policy_st
	),
	union_data as
	(
	select finc_clm_policy_pgm as company,
	CASE WHEN clm.finc_clm_lob = 'PPA' THEN 'PA' ELSE clm.finc_clm_lob END as policy_type,
	clm.finc_clm_policy_st as policy_state,
	CASE WHEN cov.cov_stat_code like '[0-9]' THEN cov.cov_code
		--WHEN cov.cov_stat_code ='LIA' and cov.cov_code in ('BI','PD') THEN cov.cov_stat_code
		WHEN cov.cov_stat_code ='LIA' THEN cov.cov_code
		ELSE cov.cov_stat_code
		END AS coverage,
	0 as grosswritten,
	0 as netwritten, 
	0 as commission,
	sum(paid_loss) as paid_loss,
	sum(paid_expense) as paid_expense

	from claims_data clm
	join Coverage_codes cov
	on clm.finc_clmc_cov_code = cov.cov_code

	group by finc_clm_policy_pgm,finc_clm_lob, finc_clm_policy_st, 
	CASE WHEN cov.cov_stat_code like '[0-9]' THEN cov.cov_code
		--WHEN cov.cov_stat_code ='LIA' and cov.cov_code in ('BI','PD') THEN cov.cov_stat_code
		WHEN cov.cov_stat_code ='LIA' THEN cov.cov_code
		ELSE cov.cov_stat_code
		END


	union all

	select company,
	policy_type,
	policy_state,
	CASE WHEN p.coverage = 'UMBI' THEN 'UMB'
		WHEN p.coverage = 'UMPD' THEN 'UMP'
		--WHEN p.coverage in ('BI','PD') THEN 'LIA'
		ELSE p.coverage END as coverage,
	sum(grosswritten_by_policy_cov) as grosswritten,
	sum(netwritten) as netwritten,
	sum(commission) as commission,
	0 as paid_loss,
	0 as paid_expense

	from policy_data p

	group by company,policy_state, policy_type,
	CASE WHEN p.coverage = 'UMBI' THEN 'UMB'
		WHEN p.coverage = 'UMPD' THEN 'UMP'
		--WHEN p.coverage in ('BI','PD') THEN 'LIA'
		ELSE p.coverage END
	)

	select concat(date_format(cast('{start_date}' as date),'MMMM'),' ',YEAR('{start_date}')) as reporting_period,
	date_format('{rundate}','MM/dd/yyyy') as rundate,
	SUBSTRING('{runtime}',1,8) as runtime,
	gcomp.gcp_desc as company,
	CONCAT(policy_type,' - ',glob.gcp_desc) as policy_type,
	policy_state,
	coverage,
	COALESCE(gcov.gcp_desc,cov.cov_desc) as coverage_desc,
	sum(grosswritten) as grosswritten,
	sum(netwritten) as netwritten,
	sum(commission) as commission,
	sum (paid_loss) as paid_loss,
	sum(paid_expense) as paid_expense

	from union_data un

	left join general_codes gcomp
	on gcomp.gcp_code=un.company
	and gcomp.gcp_internal_type='COMPANY'

	left join Coverage_codes cov
	on un.coverage = cov.cov_code

	left join general_codes gcov
	on gcov.gcp_code=un.coverage
	and gcov.gcp_internal_type='GROUP_COVERAGE'

	left join general_codes glob
	on glob.gcp_code = CASE WHEN un.policy_type = 'PA' THEN 'PPA' ELSE un.policy_type END
	and glob.gcp_internal_type='LOB'

	group by gcomp.gcp_desc,CONCAT(policy_type,' - ',glob.gcp_desc), policy_state, un.coverage,
	COALESCE(gcov.gcp_desc,cov.cov_desc)
	'''

	# Execute the query
	result = spark.sql(query)
	return result

StatementMeta(, 9ad55c41-31ed-41ee-b2e3-cd10747e3bab, 41, Finished, Available, Finished, False)

In [40]:
#Initialising List to and looping through months to store aggregated data
df_combined = None

for n in range(0, 13):  # run for 0 to 12 months ago
    start_date = (now_et.replace(day=1) - relativedelta(months=n)).strftime('%Y-%m-%d')
    end_date = (
        (now_et.replace(day=1) - relativedelta(months=n - 1)).strftime('%Y-%m-%d')
        if n != 0 else now_et.strftime('%Y-%m-%d')
    )

    print("Start Date:", start_date)
    print("End Date:", end_date)

    df = run_monthly_process(start_date, end_date)
    df.show()

    if df_combined is None:
        df_combined = df
    else:
        df_combined = df_combined.unionByName(df)

df_combined.show()
print(f"Number of rows:",df_combined.count()) 

StatementMeta(, 9ad55c41-31ed-41ee-b2e3-cd10747e3bab, 42, Finished, Available, Finished, False)

Start Date: 2026-04-01
End Date: 2026-04-06
+----------------+----------+--------+----------------+------------------+------------+--------+---------------+------------------+------------------+------------------+---------+------------+
|reporting_period|   rundate| runtime|         company|       policy_type|policy_state|coverage|  coverage_desc|      grosswritten|        netwritten|        commission|paid_loss|paid_expense|
+----------------+----------+--------+----------------+------------------+------------+--------+---------------+------------------+------------------+------------------+---------+------------+
|      April 2026|04/06/2026|15:40:38|Southern General|PA - Personal Auto|          GA|     UMB|   UNINSURED-BI|20404.040000000005|           16970.5|1909.8142999999998|     0.00|        0.00|
|      April 2026|04/06/2026|15:40:38|Southern General|PA - Personal Auto|          GA|      MP|Medical Payment|1506.5000000000002|1209.4600000000003|120.80170000000001|     0.00|     

display(result)

In [41]:
df=df_combined.toPandas()
datatypesinpandas=df.dtypes
datatypesinpandas
df['grosswritten']=df['grosswritten'].astype(float).round(2)
df['netwritten']=df['netwritten'].astype(float).round(2)
df['commission']=df['commission'].astype(float).round(2)
df['paid_loss']=df['paid_loss'].astype(float).round(2)
df['paid_expense']=df['paid_expense'].astype(float).round(2)

StatementMeta(, 9ad55c41-31ed-41ee-b2e3-cd10747e3bab, 43, Finished, Available, Finished, False)

In [42]:
schema  = StructType([
StructField("reporting_period", StringType(), True),
StructField("rundate",StringType(), True),
StructField("runtime",StringType(), True),
StructField("company", StringType(), True),
StructField("policy_type", StringType(), True),
StructField("policy_state", StringType(), True),
StructField("coverage", StringType(), True),
StructField("coverage_desc", StringType(), True),
StructField("grosswritten", DoubleType(), True),
StructField("netwritten", DoubleType(), True),
StructField("commission", DoubleType(), True),
StructField("paid_loss", DoubleType(), True),
StructField("paid_expense", DoubleType(), True),
])  
try:
    df_spark=spark.createDataFrame(df,schema)
except:

    df_spark=spark.createDataFrame([],schema)
df_spark.show()    

StatementMeta(, 9ad55c41-31ed-41ee-b2e3-cd10747e3bab, 44, Finished, Available, Finished, False)

+----------------+----------+--------+----------------+------------------+------------+--------+---------------+------------+----------+----------+---------+------------+
|reporting_period|   rundate| runtime|         company|       policy_type|policy_state|coverage|  coverage_desc|grosswritten|netwritten|commission|paid_loss|paid_expense|
+----------------+----------+--------+----------------+------------------+------------+--------+---------------+------------+----------+----------+---------+------------+
|      April 2026|04/06/2026|15:40:38|Southern General|PA - Personal Auto|          GA|     UMB|   UNINSURED-BI|    20404.04|   16970.5|   1909.81|      0.0|         0.0|
|      April 2026|04/06/2026|15:40:38|Southern General|PA - Personal Auto|          GA|      MP|Medical Payment|      1506.5|   1209.46|     120.8|      0.0|         0.0|
|      April 2026|04/06/2026|15:40:38|Southern General|PA - Personal Auto|          GA|     TOW| TOWING & LABOR|      318.33|    258.98|     30.5

In [43]:
df_spark.write.mode('overwrite').format('delta').option("overwriteSchema", "true").save('abfss://PROD_IHv2@onelake.dfs.fabric.microsoft.com/PROD_DIEP2_IH.Lakehouse/Tables/SR23205C')

StatementMeta(, 9ad55c41-31ed-41ee-b2e3-cd10747e3bab, 45, Finished, Available, Finished, False)